# LAB | API Calling with JSON Extra

## Scenario

You're continuing your work on the automated product listing generator. Clients now want to provide product information via JSON API requests instead of only using images. Your task is to build a robust system that validates incoming JSON data using Pydantic, ensures data quality, and processes client requests reliably.

This lab builds on your previous ChatGPT API work and adds data validation capabilities.

## Learning Objectives

- Understand JSON schema validation
- Use Pydantic for data validation
- Create Pydantic models for structured data
- Validate API request payloads
- Handle validation errors gracefully
- Integrate validation with API processing
- Build a complete API workflow with validation

**Estimated Time:** 120-150 minutes

## Prerequisites

- Completion of Lab M1.05 (API Calling to ChatGPT)
- Understanding of JSON data format
- Basic knowledge of Python classes
- Familiarity with error handling

## Environment Setup

Run the next cell to install the required libraries into the same Python environment used by this notebook kernel.


In [ ]:
%pip install -r requirements.txt


## Generated Artifacts

This notebook writes sample and output files into `data/` and writes logs into `logs/` when applicable. Those folders contain generated artifacts, not hand-written source files.


## What You'll Build

- Pydantic models for product data validation
- A JSON validation system
- Integration with your existing ChatGPT API workflow
- Error handling for invalid data
- A complete validated API pipeline

### Why this matters

In real-world applications, you cannot assume incoming data is correct. Validation prevents errors, improves security, and ensures data quality. Pydantic is a widely used Python library for validation and is commonly used in frameworks like FastAPI.

## Success Criteria

- Create Pydantic models for product data
- Validate JSON input successfully
- Handle validation errors appropriately
- Integrate validation with API processing
- Process validated requests end-to-end

## Background Story

Your product listing generator is working well, but now clients want to send product information via API requests. However, some clients send incomplete data, wrong data types, missing fields, or invalid values such as negative prices and empty names.

You need to validate all incoming data before processing to ensure:

- Data quality and consistency
- System reliability
- Better error messages for clients
- Prevention of API errors downstream

Pydantic will help you create validation rules and automatically check all incoming data.

# Step 1: Setting Up Pydantic

**Objective:** Install Pydantic and understand its basic usage.

### What to do

- Install Pydantic
- Create a simple validation example
- Understand how Pydantic works

### Required package

```bash
pip install pydantic
```

### Suggested starter ideas

- Create a simple `BaseModel`
- Add fields like `name`, `price`, and `quantity`
- Add validators for positive numeric values

### Expected outcome

You should see how Pydantic validates data and raises errors for invalid input.

### Checkpoint

Verify that Pydantic is installed and basic validation works.

In [ ]:
from pydantic import BaseModel, Field, ValidationError, field_validator
from openai import OpenAI
from typing import Optional, List
from datetime import datetime
from pathlib import Path
import json
import os

print("=" * 60)
print("STEP 1 | PYDANTIC BASICS")
print("=" * 60)

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)


class SimpleProduct(BaseModel):
    """Small example model to demonstrate validation."""

    name: str = Field(min_length=2)
    price: float = Field(gt=0)
    quantity: int = Field(default=1, ge=1)

    @field_validator("name")
    @classmethod
    def clean_name(cls, value: str) -> str:
        value = value.strip()
        if not value:
            raise ValueError("Name cannot be empty")
        return value


print("\n1. Valid data:")
try:
    product1 = SimpleProduct(name="Widget", price=10.99, quantity=5)
    print("  ✓ Valid:", product1.model_dump())
except ValidationError as error:
    print("  ✗ Error:", error)

print("\n2. Invalid data (negative price):")
try:
    product2 = SimpleProduct(name="Widget", price=-10.99)
    print(product2)
except ValidationError as error:
    print("  ✗ Validation error (expected):")
    print(error)

print("\n3. Invalid data (blank name):")
try:
    product3 = SimpleProduct(name="   ", price=15.00)
    print(product3)
except ValidationError as error:
    print("  ✗ Validation error (expected):")
    print(error)

print("\n✓ Pydantic basics working!")


# Step 2: Creating Product Data Models

**Objective:** Design Pydantic models for product listing requests.

### What to do

- Define the structure of product data for your previous listing generator
- Create Pydantic models with validation rules that fit your use case
- Add custom validators for business logic

### Tips

- Consider fields like `name`, `price`, `category`, `description`, and `additional_info`
- Start simple and add complexity gradually
- Think about which fields are required and which can be optional

### Expected outcome

You should have working Pydantic models that validate product data correctly.

### Checkpoint

Verify that:

- Valid data passes validation
- Invalid data raises appropriate errors
- Required fields are enforced
- Custom validators work correctly

In [ ]:
print("=" * 60)
print("STEP 2 | PRODUCT REQUEST MODEL")
print("=" * 60)


class ProductRequest(BaseModel):
    """Validated input schema for product listing requests."""

    name: str = Field(min_length=3, max_length=80)
    category: str = Field(min_length=3, max_length=40)
    price: float = Field(gt=0)
    description: str = Field(min_length=15, max_length=500)
    quantity: int = Field(default=1, ge=1)
    currency: str = Field(default="USD", min_length=3, max_length=3)
    brand: Optional[str] = Field(default=None, max_length=40)
    features: List[str] = Field(default_factory=list)
    additional_info: Optional[str] = None

    @field_validator("name", "category", "description", "brand", "additional_info")
    @classmethod
    def strip_text_fields(cls, value):
        if value is None:
            return value
        value = value.strip()
        if value == "":
            raise ValueError("Text fields cannot be empty")
        return value

    @field_validator("currency")
    @classmethod
    def validate_currency(cls, value: str) -> str:
        value = value.upper().strip()
        if len(value) != 3 or not value.isalpha():
            raise ValueError("Currency must be a 3-letter code such as USD or EUR")
        return value

    @field_validator("features")
    @classmethod
    def validate_features(cls, value: List[str]) -> List[str]:
        cleaned = [feature.strip() for feature in value if feature.strip()]
        if not cleaned:
            raise ValueError("At least one feature is required")
        return cleaned


valid_product_data = {
    "name": "EcoSmart Water Bottle",
    "category": "Kitchen",
    "price": 24.99,
    "description": "Reusable stainless steel bottle with double-wall insulation that keeps drinks cold for 24 hours.",
    "quantity": 12,
    "currency": "usd",
    "brand": "EcoSmart",
    "features": ["BPA-free", "Leak-proof lid", "750ml capacity"],
    "additional_info": "Available in blue, black, and silver."
}

invalid_product_data = {
    "name": "  ",
    "category": "Ki",
    "price": -5,
    "description": "Too short",
    "quantity": 0,
    "currency": "dollars",
    "features": ["   ", ""]
}

print("\nValid product test:")
try:
    valid_product = ProductRequest.model_validate(valid_product_data)
    print("  ✓ Passed validation")
    print(valid_product.model_dump())
except ValidationError as error:
    print(error)

print("\nInvalid product test:")
try:
    invalid_product = ProductRequest.model_validate(invalid_product_data)
    print(invalid_product)
except ValidationError as error:
    print("  ✗ Validation errors found:")
    for item in error.errors():
        print(f"  - {item['loc']}: {item['msg']}")


# Step 3: Validating JSON Input

**Objective:** Validate JSON data from files.

### What to do

- Generate example JSON files using the structure from Step 2
- Create both valid and invalid examples
- Write a function to load JSON data and validate it using your Pydantic models
- Define a strategy to handle validation errors
- Return validated data when successful

### Expected outcome

You should be able to validate JSON data and get clear error messages for invalid input.

### Checkpoint

Verify that validation works for both valid and invalid JSON.

In [ ]:
print("=" * 60)
print("STEP 3 | JSON INPUT VALIDATION")
print("=" * 60)


valid_payload = {
    "name": "EcoSmart Water Bottle",
    "category": "Kitchen",
    "price": 24.99,
    "description": "Reusable stainless steel bottle with double-wall insulation that keeps drinks cold for 24 hours.",
    "quantity": 12,
    "currency": "USD",
    "brand": "EcoSmart",
    "features": ["BPA-free", "Leak-proof lid", "750ml capacity"],
    "additional_info": "Ideal for travel, work, and gym sessions."
}

invalid_payload = {
    "name": "X",
    "category": "",
    "price": -12,
    "description": "Bad data",
    "quantity": -2,
    "currency": "US D",
    "features": []
}

batch_payloads = [
    valid_payload,
    invalid_payload,
    {
        "name": "Desk Lamp Pro",
        "category": "Office",
        "price": 59.50,
        "description": "Adjustable LED desk lamp with touch controls and three brightness modes for focused work.",
        "quantity": 4,
        "currency": "EUR",
        "brand": "BrightSpace",
        "features": ["LED", "Touch control", "USB charging port"],
        "additional_info": "Ships with power adapter."
    }
]

with open(DATA_DIR / "valid_product.json", "w", encoding="utf-8") as file:
    json.dump(valid_payload, file, indent=2)

with open(DATA_DIR / "invalid_product.json", "w", encoding="utf-8") as file:
    json.dump(invalid_payload, file, indent=2)

with open(DATA_DIR / "batch_requests.json", "w", encoding="utf-8") as file:
    json.dump(batch_payloads, file, indent=2)

print("Created JSON example files in data/: data/valid_product.json, data/invalid_product.json, data/batch_requests.json")


def validate_payload(payload: dict, model: type[BaseModel]) -> dict:
    """Validate a Python dictionary against a Pydantic model."""

    try:
        validated = model.model_validate(payload)
        return {"ok": True, "data": validated, "errors": []}
    except ValidationError as error:
        return {"ok": False, "data": None, "errors": error.errors()}



def load_and_validate_json(file_path: str, model: type[BaseModel]) -> dict:
    """Load JSON from disk, then validate its contents."""

    try:
        with open(file_path, "r", encoding="utf-8") as file:
            payload = json.load(file)
    except FileNotFoundError:
        return {"ok": False, "data": None, "errors": [{"msg": f"File not found: {file_path}"}]}
    except json.JSONDecodeError as error:
        return {"ok": False, "data": None, "errors": [{"msg": f"Invalid JSON: {error}"}]}

    return validate_payload(payload, model)


print("\nValid JSON file result:")
valid_result = load_and_validate_json(DATA_DIR / "valid_product.json", ProductRequest)
print(valid_result["data"].model_dump() if valid_result["ok"] else valid_result["errors"])

print("\nInvalid JSON file result:")
invalid_result = load_and_validate_json(DATA_DIR / "invalid_product.json", ProductRequest)
print(invalid_result["errors"])


# Step 4: Integrating with ChatGPT API

**Objective:** Combine validation with your existing ChatGPT API workflow.

### What to do

- Import or reuse your previous ChatGPT/OpenAI API code
- Validate input before processing
- Process only validated requests
- Return appropriate responses
- Validate that the output is also in a standardized format, for example with Pydantic models

### Expected outcome

You should have a complete workflow that validates the input before processing and validates the output after processing.

### Checkpoint

Verify that:

- Invalid data is rejected before API calls
- Valid data is processed successfully
- Errors are handled gracefully

In [ ]:
print("=" * 60)
print("STEP 4 | OPENAI INTEGRATION")
print("=" * 60)


class ProductListing(BaseModel):
    """Structured output for the generated product listing."""

    title: str
    marketing_description: str
    bullet_points: List[str]
    tags: List[str]
    category: str
    price_label: str


api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key) if api_key else None



def build_listing_prompt(product: ProductRequest) -> str:
    features_text = ", ".join(product.features)
    brand_text = product.brand or "Unknown brand"
    extra_text = product.additional_info or "No additional information provided."

    return (
        f"Create a polished e-commerce product listing for this item. "
        f"Name: {product.name}. "
        f"Category: {product.category}. "
        f"Brand: {brand_text}. "
        f"Price: {product.price} {product.currency}. "
        f"Description: {product.description}. "
        f"Features: {features_text}. "
        f"Additional info: {extra_text}."
    )



def generate_product_listing(product: ProductRequest) -> ProductListing:
    """Generate a validated product listing using OpenAI when available."""

    if client is None:
        print("OPENAI_API_KEY not found. Using a local demo response instead of a live API call.")
        return ProductListing(
            title=f"{product.brand + ' ' if product.brand else ''}{product.name}",
            marketing_description=(
                f"The {product.name} is a {product.category.lower()} product designed for customers who want "
                f"reliable performance, thoughtful design, and practical everyday value."
            ),
            bullet_points=product.features[:3],
            tags=[product.category.lower(), product.currency.lower(), "validated-product"],
            category=product.category,
            price_label=f"{product.currency} {product.price:.2f}"
        )

    completion = client.chat.completions.parse(
        model="gpt-4o-2024-08-06",
        messages=[
            {
                "role": "system",
                "content": (
                    "You create concise, high-quality e-commerce listings. "
                    "Return structured data only."
                )
            },
            {
                "role": "user",
                "content": build_listing_prompt(product)
            }
        ],
        response_format=ProductListing,
    )

    return completion.choices[0].message.parsed


validated_product = load_and_validate_json(DATA_DIR / "valid_product.json", ProductRequest)

if validated_product["ok"]:
    listing = generate_product_listing(validated_product["data"])
    print("\nStructured listing output:")
    print(listing.model_dump_json(indent=2))
else:
    print("Input validation failed. API call skipped.")
    print(validated_product["errors"])


# Step 5: Handling Multiple Requests

**Objective:** Process a list of incoming JSON requests in one run.

### What to do

- Load a list of request payloads from a JSON file
- Validate each payload with your Pydantic model
- Process valid requests
- Collect structured errors for invalid ones
- Return a summary of successes and failures

### Expected outcome

You can process batches while preserving clear per-request validation feedback.

### Checkpoint

Verify that your batch run handles mixed valid and invalid requests without stopping the full pipeline.

In [ ]:
print("=" * 60)
print("STEP 5 | BATCH REQUEST PROCESSING")
print("=" * 60)



def process_batch_requests(file_path: str) -> dict:
    """Validate and process a batch of product requests."""

    try:
        with open(file_path, "r", encoding="utf-8") as file:
            payloads = json.load(file)
    except FileNotFoundError:
        return {"total": 0, "success_count": 0, "error_count": 1, "results": [{"status": "error", "errors": ["Batch file not found"]}]}
    except json.JSONDecodeError as error:
        return {"total": 0, "success_count": 0, "error_count": 1, "results": [{"status": "error", "errors": [f"Invalid JSON: {error}"]}]}

    results = []

    for index, payload in enumerate(payloads, start=1):
        validation = validate_payload(payload, ProductRequest)

        if not validation["ok"]:
            results.append({
                "index": index,
                "status": "validation_error",
                "errors": validation["errors"]
            })
            continue

        listing = generate_product_listing(validation["data"])
        results.append({
            "index": index,
            "status": "success",
            "input": validation["data"].model_dump(),
            "output": listing.model_dump()
        })

    success_count = sum(1 for item in results if item["status"] == "success")
    error_count = len(results) - success_count

    return {
        "total": len(results),
        "success_count": success_count,
        "error_count": error_count,
        "results": results
    }


batch_summary = process_batch_requests(DATA_DIR / "batch_requests.json")
print(json.dumps({
    "total": batch_summary["total"],
    "success_count": batch_summary["success_count"],
    "error_count": batch_summary["error_count"]
}, indent=2))

print("\nDetailed first result:")
print(json.dumps(batch_summary["results"][0], indent=2, default=str))


# Step 6: Creating a Client Request Handler

**Objective:** Wrap validation and processing into a reusable handler function.

### What to do

- Create a `handle_client_request(payload: dict)` function
- Validate input
- Call your processing logic
- Validate output
- Return a consistent response shape for success and error cases

### Expected outcome

You have one reusable entry point for future API endpoints or scripts.

### Checkpoint

Verify that the handler returns predictable JSON for both valid and invalid payloads.

In [ ]:
print("=" * 60)
print("STEP 6 | CLIENT REQUEST HANDLER")
print("=" * 60)



def handle_client_request(payload: dict) -> dict:
    """Single entry point for validating input, generating output, and returning a consistent response."""

    validation = validate_payload(payload, ProductRequest)

    if not validation["ok"]:
        return {
            "status": "error",
            "message": "Input validation failed",
            "data": None,
            "errors": validation["errors"]
        }

    listing = generate_product_listing(validation["data"])

    return {
        "status": "success",
        "message": "Product listing generated successfully",
        "data": listing.model_dump(),
        "errors": []
    }


print("\nHandler with valid payload:")
handler_success = handle_client_request(valid_payload)
print(json.dumps(handler_success, indent=2))

print("\nHandler with invalid payload:")
handler_error = handle_client_request(invalid_payload)
print(json.dumps(handler_error, indent=2))


# Bonus Challenge: New Dataset from Scratch

**Objective:** Apply your validation skills to a completely new dataset.

### Task

Choose a new dataset from Kaggle, Hugging Face, or create your own. Then:

- Analyze the dataset structure
- Identify field names, types, and sensible validation rules
- Create Pydantic models for the dataset
- Build a validation pipeline
- Validate each record and report results
- Use validated data for some purpose such as analysis or API calls
- Integrate validated data with ChatGPT/OpenAI
- Validate the model output as well

### Suggested datasets

- Customer Reviews Dataset
- Real Estate Listings
- Employee Records
- E-commerce Orders

In [ ]:
print("=" * 60)
print("BONUS | NEW DATASET VALIDATION")
print("=" * 60)


class CustomerReview(BaseModel):
    review_id: str = Field(min_length=3)
    product_name: str = Field(min_length=3)
    rating: int = Field(ge=1, le=5)
    review_text: str = Field(min_length=10, max_length=500)
    review_date: str
    verified_purchase: bool = False

    @field_validator("review_id", "product_name", "review_text")
    @classmethod
    def strip_review_text(cls, value: str) -> str:
        value = value.strip()
        if not value:
            raise ValueError("This field cannot be blank")
        return value

    @field_validator("review_date")
    @classmethod
    def validate_date(cls, value: str) -> str:
        datetime.strptime(value, "%Y-%m-%d")
        return value


class ReviewAnalysis(BaseModel):
    sentiment: str
    short_summary: str
    needs_follow_up: bool


review_dataset = [
    {
        "review_id": "R-1001",
        "product_name": "EcoSmart Water Bottle",
        "rating": 5,
        "review_text": "Excellent insulation and very durable. I use it every day at work.",
        "review_date": "2026-04-20",
        "verified_purchase": True
    },
    {
        "review_id": "R-1002",
        "product_name": "Desk Lamp Pro",
        "rating": 0,
        "review_text": "Bad",
        "review_date": "2026/04/21",
        "verified_purchase": False
    }
]



def validate_reviews(records: List[dict]) -> dict:
    valid_reviews = []
    invalid_reviews = []

    for index, record in enumerate(records, start=1):
        try:
            review = CustomerReview.model_validate(record)
            valid_reviews.append(review)
        except ValidationError as error:
            invalid_reviews.append({"index": index, "errors": error.errors()})

    return {
        "total": len(records),
        "valid_count": len(valid_reviews),
        "invalid_count": len(invalid_reviews),
        "valid_reviews": valid_reviews,
        "invalid_reviews": invalid_reviews
    }



def analyze_review(review: CustomerReview) -> ReviewAnalysis:
    if client is None:
        sentiment = "positive" if review.rating >= 4 else "negative" if review.rating <= 2 else "neutral"
        return ReviewAnalysis(
            sentiment=sentiment,
            short_summary=review.review_text[:80],
            needs_follow_up=review.rating <= 2
        )

    completion = client.chat.completions.parse(
        model="gpt-4o-2024-08-06",
        messages=[
            {
                "role": "system",
                "content": "Analyze customer reviews and return structured sentiment data only."
            },
            {
                "role": "user",
                "content": (
                    f"Review the following customer feedback. Product: {review.product_name}. "
                    f"Rating: {review.rating}/5. Text: {review.review_text}"
                )
            }
        ],
        response_format=ReviewAnalysis,
    )

    return completion.choices[0].message.parsed


review_results = validate_reviews(review_dataset)
print(json.dumps({
    "total": review_results["total"],
    "valid_count": review_results["valid_count"],
    "invalid_count": review_results["invalid_count"]
}, indent=2))

if review_results["valid_reviews"]:
    review_analysis = analyze_review(review_results["valid_reviews"][0])
    print("\nExample structured review analysis:")
    print(review_analysis.model_dump_json(indent=2))

print("\nInvalid review records:")
print(json.dumps(review_results["invalid_reviews"], indent=2))


# Deliverables

## Required

- Your complete `api_json_validation.py` file with:
  - Pydantic models
  - Validation functions
  - Integration with ChatGPT API
  - Client request handler
- Example JSON files with valid and invalid examples
- Screenshots or output showing:
  - Successful validation
  - Validation errors
  - Complete workflow execution
- A brief report including:
  - How Pydantic validation works
  - Benefits of validation
  - Challenges you faced
  - Improvements you made

## Bonus

- Pydantic models for a new dataset
- Validation results and statistics
- A brief analysis report

# Troubleshooting Notes

## Common issues

### 1. Validation errors are unclear
- Use `try/except` to catch `ValidationError`
- Use `error.errors()` for detailed error information

### 2. Optional fields are required
- Use `Optional[type] = None`
- Or use `Field(default=None)`

### 3. Custom validator is not working
- Check the `@validator('field_name')` decorator
- Check the validator function signature
- Use `pre=True` if needed before type conversion

### 4. JSON parsing fails before validation
- Validate raw JSON format first with `json.loads()`
- Handle `JSONDecodeError` separately

### 5. Validation is too strict or too lenient
- Adjust `Field` constraints such as `min_length`, `max_length`, `gt`, and `lt`
- Modify custom validators as needed

### 6. Integration with existing code is difficult
- Convert Pydantic models to dictionaries with `.dict()`
- Use `.json()` for serialization
- Access fields as attributes such as `product.name`

# To-Do Checklist

- Complete Step 1: Setting Up Pydantic
- Complete Step 2: Creating Product Data Models
- Complete Step 3: Validating JSON Input
- Complete Step 4: Integrating with ChatGPT API
- Complete Step 5: Handling Multiple Requests
- Complete Step 6: Creating a Client Request Handler
- Test with various valid and invalid inputs
- Document your validation rules
- Optionally complete the bonus challenge
- Submit your work

# Reflection

After completing the lab, reflect on the following:

- Why is validation important before processing data?
- What advantages does Pydantic provide over manual validation?
- How does validation improve error messages and debugging?
- How do type hints and Pydantic work together?
- How does validation improve API reliability?
- Which validation patterns were most useful?

## Key takeaways

- Validation prevents errors and improves data quality
- Pydantic provides automatic validation with clear error messages
- Type hints make code more maintainable and less error-prone
- Validating early saves time and resources
- Good validation improves user experience
- Validation is essential for production systems